In [1]:
import pandas as pd
import numpy as np
import os
import cv2
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.utils import img_to_array, to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPool2D, Flatten, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Suppress warnings
warnings.filterwarnings('ignore')

# Constants
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 100
TEST_SIZE = 0.2
MODEL_SAVE_PATH = 'best_model.h5'

def load_data():
    """Load and preprocess the dataset."""
    train = pd.DataFrame(columns=["label", "path"])
    test = pd.DataFrame(columns=["label", "path"])

    for dirname, _, filenames in os.walk('/kaggle/input/grape-disease-dataset-original/Original Data'):
        for filename in filenames:
            paths = dirname.split('/')
            batch = paths[-2]
            label = paths[-1]
            file_path = os.path.join(dirname, filename)
            
            dic = {'label': label, 'path': file_path}
            if batch == 'test':
                test.loc[len(test)] = dic
            elif batch == 'train':
                train.loc[len(train)] = dic
    
    return train, test

def preprocess_images(df):
    """Preprocess images from dataframe."""
    images = []
    for i, path in enumerate(df['path']):
        image = cv2.imread(path)
        image = cv2.resize(image, IMAGE_SIZE)
        image = img_to_array(image)
        images.append(image)
        
        if i % 500 == 0:
            print(f"[INFO] Processed {i} images")
    
    images = np.array(images) / 255.0
    return images

def encode_labels(y):
    """Encode labels to categorical format."""
    disease_list = y.unique()
    disease_dict = {disease: i for i, disease in enumerate(disease_list)}
    y_encoded = np.array([disease_dict[label] for label in y])
    return to_categorical(y_encoded, num_classes=len(disease_list)), disease_dict

def build_cnn_model():
    """Build and compile the CNN model."""
    model = Sequential([
        Conv2D(32, (5,5), padding='Same', activation='relu', input_shape=(224,224,3)),
        Conv2D(32, (5,5), padding='Same', activation='relu'),
        BatchNormalization(momentum=0.15),
        MaxPool2D(pool_size=(2,2)),
        Dropout(0.25),
        
        Conv2D(64, (3,3), padding='Same', activation='relu'),
        Conv2D(64, (3,3), padding='Same', activation='relu'),
        BatchNormalization(momentum=0.15),
        MaxPool2D(pool_size=(2,2), strides=(2,2)),
        Dropout(0.25),
        
        Conv2D(32, (5,5), padding='Same', activation='relu'),
        Conv2D(32, (5,5), padding='Same', activation='relu'),
        BatchNormalization(momentum=0.15),
        MaxPool2D(pool_size=(2,2)),
        Dropout(0.25),
        
        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.4),
        Dense(4, activation="softmax")
    ])
    
    optimizer = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def evaluate_ml_models(X_train, y_train, X_test, y_test):
    """Evaluate various ML models on CNN features."""
    models = {
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "XGBoost": XGBClassifier(),
        "KNN": KNeighborsClassifier()
    }
    
    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results[name] = acc
        print(f"\n{name} Classification Report:")
        print(classification_report(y_test, y_pred))
    
    return results

def main():
    # Load and preprocess data
    train_df, test_df = load_data()
    X_train = preprocess_images(train_df)
    y_train, label_dict = encode_labels(train_df['label'])
    
    # Split data
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=TEST_SIZE, stratify=y_train
    )
    
    # Build and train CNN model
    model = build_cnn_model()
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor='val_accuracy')
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks
    )
    
    # Evaluate CNN model
    loss, accuracy = model.evaluate(X_val, y_val)
    print(f"\nCNN Model Evaluation:")
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    
    # Generate classification report for CNN
    y_pred = model.predict(X_val)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_val, axis=1)
    
    print("\nCNN Classification Report:")
    print(classification_report(y_true_classes, y_pred_classes, target_names=label_dict.keys()))
    
    # Extract CNN features and evaluate ML models
    cnn_features_train = model.predict(X_train)
    cnn_features_val = model.predict(X_val)
    
    print("\nEvaluating ML models on CNN features...")
    ml_results = evaluate_ml_models(
        cnn_features_train, np.argmax(y_train, axis=1),
        cnn_features_val, np.argmax(y_val, axis=1)
    )
    
    print("\nML Models Accuracy Comparison:")
    for name, acc in ml_results.items():
        print(f"{name}: {acc:.4f}")

if __name__ == "__main__":
    main()

2025-08-03 05:52:05.208403: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754200325.399440      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754200325.450376      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[INFO] Processed 0 images
[INFO] Processed 500 images
[INFO] Processed 1000 images
[INFO] Processed 1500 images
[INFO] Processed 2000 images
[INFO] Processed 2500 images
[INFO] Processed 3000 images
[INFO] Processed 3500 images
[INFO] Processed 4000 images
[INFO] Processed 4500 images
[INFO] Processed 5000 images
[INFO] Processed 5500 images
[INFO] Processed 6000 images
[INFO] Processed 6500 images
[INFO] Processed 7000 images


I0000 00:00:1754200405.464846      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/100


I0000 00:00:1754200420.458802      57 service.cc:148] XLA service 0x79dff40057f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1754200420.459430      57 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1754200421.014214      57 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-08-03 05:53:48.899381: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng0{} for conv (f32[32,32,5,5]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,32,224,224]{3,2,1,0}, f32[32,32,224,224]{3,2,1,0}), window={size=5x5 pad=2_2x2_2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-08-03 05:53:49.509847: E external/l

  2/181 ━━━━━━━━━━━━━━━━━━━━ 14s 80ms/step - accuracy: 0.2578 - loss: 11.9414

I0000 00:00:1754200434.237410      57 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


181/181 ━━━━━━━━━━━━━━━━━━━━ 48s 160ms/step - accuracy: 0.6575 - loss: 3.2774 - val_accuracy: 0.7536 - val_loss: 0.6427
Epoch 2/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.7983 - loss: 0.4848 - val_accuracy: 0.8221 - val_loss: 0.4522
Epoch 3/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.8464 - loss: 0.3967 - val_accuracy: 0.8408 - val_loss: 0.4719
Epoch 4/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 14s 79ms/step - accuracy: 0.8647 - loss: 0.3631 - val_accuracy: 0.6990 - val_loss: 0.9056
Epoch 5/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 14s 79ms/step - accuracy: 0.8965 - loss: 0.2670 - val_accuracy: 0.7792 - val_loss: 0.7026
Epoch 6/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.9293 - loss: 0.2204 - val_accuracy: 0.9163 - val_loss: 0.2743
Epoch 7/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.9323 - loss: 0.1956 - val_accuracy: 0.9225 - val_loss: 0.3065
Epoch 8/100
181/181 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.9445 - loss: 0.1609 - val